In [ ]:
import catboost
from catboost import CatBoostClassifier, CatBoostRegressor
import catboost.datasets

import math
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import shap

shap.initjs()

In [ ]:
X, y = shap.datasets.california(n_points=500)
df = X.copy()
df["MedHouseVal"] = y
df.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)

X_train.shape, X_test.shape

In [ ]:
num = X.select_dtypes(include="number")
if num.shape[1] == 0:
    raise ValueError("X has no numeric columns to plot.")

max_cols_per_fig = 12
col_wrap = 4

for start in range(0, num.shape[1], max_cols_per_fig):
    cols = num.columns[start : start + max_cols_per_fig]
    long_df = num[cols].melt(var_name="feature", value_name="value")

    g = sns.catplot(
        data=long_df,
        x="value",
        # y="feature",
        col="feature",
        kind="box",
        col_wrap=col_wrap,
        sharex=False,
        sharey=False,
        height=2.8,
        aspect=1.1,
    )

In [ ]:
sns.histplot(df["MedHouseVal"], bins=30, kde=True)

In [ ]:
norm = mpl.colors.Normalize(vmin=df["MedHouseVal"].min(), vmax=df["MedHouseVal"].max())

ax = sns.scatterplot(
    data=df,
    x="Longitude",
    y="Latitude",
    hue="MedHouseVal",
    palette="viridis",
    hue_norm=norm,
    legend=False,
    s=25,
    alpha=0.8,
    edgecolor=None,
)
ax.set_title("California locations colored by median house value")

sm = mpl.cm.ScalarMappable(cmap="viridis", norm=norm)
sm.set_array([])
ax.figure.colorbar(sm, ax=ax, label="MedHouseVal")

In [ ]:
model = CatBoostRegressor(iterations=300, learning_rate=0.1, random_seed=123)
model.fit(X_train, y_train, verbose=False, plot=False)

pred_train = model.predict(X_train)
pred_test = model.predict(X_test)

metrics = pd.DataFrame(
    {
        "split": ["train", "test"],
        "MAE": [
            mean_absolute_error(y_train, pred_train),
            mean_absolute_error(y_test, pred_test),
        ],
        "RMSE": [
            mean_squared_error(y_train, pred_train) ** 0.5,
            mean_squared_error(y_test, pred_test) ** 0.5,
        ],
        "R2": [
            r2_score(y_train, pred_train),
            r2_score(y_test, pred_test),
        ],
    }
)
metrics

In [ ]:
y_train[0]

In [ ]:
sns.heatmap(
    df.corr(), cmap="bwr", vmin=-1, vmax=1, annot=True, fmt=".2f", linewidths=0.5
)

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_train)

shap.plots.force(shap_values[0, ...])

In [ ]:
shap.plots.force(shap_values)

In [ ]:
shap.plots.beeswarm(shap_values)

In [ ]:
train_df, test_df = catboost.datasets.amazon()
y = train_df.ACTION
X = train_df.drop("ACTION", axis=1)
cat_features = list(range(0, X.shape[1]))

In [ ]:
cat_features

In [ ]:
num = X.select_dtypes(include="number")
if num.shape[1] == 0:
    raise ValueError("X has no numeric columns to plot.")

max_cols_per_fig = 12
col_wrap = 4

for start in range(0, num.shape[1], max_cols_per_fig):
    cols = num.columns[start : start + max_cols_per_fig]
    long_df = num[cols].melt(var_name="feature", value_name="value")

    g = sns.catplot(
        data=long_df,
        x="value",
        # y="feature",
        col="feature",
        kind="box",
        col_wrap=col_wrap,
        sharex=False,
        sharey=False,
        height=2.8,
        aspect=1.1,
    )